# Getting documents

Goal: turn a GitHub docs repo into a plain list of Python dicts we can search later.
First by hand, then with a library that does the same thing in three lines.

## Download the repo as a zip

GitHub hands out any branch as a zip, so one HTTP request gets us the whole repo.
No `git clone`, no local checkout to keep in sync.

In [ ]:
!uv add requests

In [ ]:
import requests

In [ ]:
repo_owner = "evidentlyai"
repo_name = "docs"
repo_branch = "main"

zip_url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{repo_branch}.zip"
zip_response = requests.get(zip_url, allow_redirects=True)
zip_response.raise_for_status()

`raise_for_status()` is the real check: a wrong owner, repo or branch comes back as a 404 HTML
page, and without it the first sign of trouble is `zipfile` raising `BadZipFile` three cells
later. The length below is just a sanity check on the size - this repo is around 17 MB.

In [ ]:
len(zip_response.content)

## Open the zip without unpacking it

`BytesIO` lets `zipfile` read straight from the bytes we just downloaded, so nothing touches disk.

In [ ]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [ ]:
zip_archive

In [ ]:
zip_archive.namelist()

## Pull out one file first

Much easier to get a single file right than to debug a loop over ninety of them.

In [ ]:
file_name = "docs-main/docs/platform/alerts.mdx"
mdx_file = zip_archive.open(file_name)

In [ ]:
with mdx_file:
    mdx_content = mdx_file.read().decode("utf-8")

## Split the frontmatter from the text

Every `.mdx` page starts with a small YAML header (title, description). `frontmatter` peels that
off, so we get the metadata as a dict and the body as clean text instead of the header bleeding
into the content we want to search.

In [ ]:
!uv add python-frontmatter

In [ ]:
import frontmatter

post = frontmatter.loads(mdx_content)
print(post.content[:500])

In [ ]:
post.metadata

In [ ]:
post.content

## Clean up the file path

The zip nests everything under `docs-main/`, which isn't part of the real path in the repo.
Split once and keep the tail.

In [ ]:
file_name.split("/", maxsplit=1)

In [ ]:
filename_corrected = file_name.split("/", maxsplit=1)[-1]

In [ ]:
filename_corrected

## Shape one document

This dict is the format we want everything in: the text to search over, plus the metadata we'd
want to show alongside a result.

In [ ]:
doc = {
    "content": post.content,
    "title": post.metadata.get("title"),
    "description": post.metadata.get("description"),
    "filename": filename_corrected,
}

In [ ]:
doc

## Run it over the whole repo

Same steps as above, now in a loop. Anything that isn't `.md` or `.mdx` gets skipped - images and
config files are just noise for search.

In [ ]:
documents = []

with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zip_ref:
    for file_path in zip_ref.namelist():
        if not file_path.endswith((".md", ".mdx")):
            continue

        with zip_ref.open(file_path) as file:
            content = file.read().decode("utf-8")
            post = frontmatter.loads(content)

            doc = {
                "content": post.content,
                "title": post.metadata.get("title"),
                "description": post.metadata.get("description"),
                "filename": file_path.split("/", 1)[-1],
            }
            documents.append(doc)

In [ ]:
len(documents)

## Same job, done by a library

`gitsource` wraps everything above: download, unzip, filter by extension, parse the frontmatter.
Worth doing it by hand once to know what it's doing for you, then use this.

In [ ]:
!uv add gitsource

In [ ]:
%pip install gitsource

In [ ]:
%pip show gitsource

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

In [ ]:
files[0]

In [ ]:
files[10].content

In [ ]:
files[10].parse()

`.parse()` gives back the same dict shape we built manually, so this one line replaces the whole
loop. Note it reuses the `documents` name, so the two cells below now read this version, not the
hand-built one.

In [ ]:
documents = [f.parse() for f in files]

In [ ]:
len(documents)

In [ ]:
documents[10]

# Keyword search

Goal: given a question, find the handful of documents worth reading. Word overlap only, no
embeddings yet - the same idea as the search box on a docs site.

## Pick a query

One fixed question to judge the results against while wiring things up.

In [ ]:
query = "LLM as a Judge"

## Install the search library

Two routes, and in this notebook they do different things. The kernel here runs the global
interpreter, not the project `.venv`, so `%pip install` is the one that actually reaches it -
that's why each `uv add` below is followed by a `%pip install`. `uv add` still earns its place: it
records the dependency in `pyproject.toml` so the environment is reproducible. Point the kernel at
the uv venv and `uv add` alone would be enough.

In [ ]:
!uv add minsearch

In [ ]:
%pip install minsearch

## Build the index

`text_fields` get tokenised and scored, `keyword_fields` are kept for exact-match filtering.
`fit` does the indexing in memory, so it has to run again any time `documents` changes.

In [ ]:
from minsearch import Index

In [ ]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

## Re-fit the index on these chunks

`document_chunks` now holds the library's chunks, so the `chunk_index` above is indexing a list
that no longer exists under that name. Add `chunk_id` back - numbering restarts per page, which
works because `chunk_documents` keeps each page's chunks together - and fit again.

## Run the search

`search` scores every document against the query and returns the matching dicts themselves, not
ids to look up - so `results[2]` is a document, with its `title` and `content` already attached.
Ten results unless fewer matched; `num_results` changes that.

In [ ]:
results = index.search(query)

In [ ]:
len(results)

In [ ]:
results[2]

# Chunking

The documents are whole pages, and the longest ones run to tens of thousands of characters. That
hurts twice: too much text to hand a model, and a hit tells us the page matched without saying
where. Splitting each page into overlapping windows gives smaller units to retrieve, with enough
overlap that an answer sitting on a boundary still lands inside a chunk.

## How long are the documents?

Worth looking before picking a window size - the top few pages set the scale we're cutting down.

In [ ]:
doc_sizes = [(doc["filename"], len(doc["content"])) for doc in documents]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

## Sliding window, on a list first

A list of numbers makes the mechanics visible: `window_size` is how much each chunk holds, `step`
is how far the window moves, and what the two leave behind is the overlap. The short tail at the
end gets dropped rather than kept as a stub.

In [ ]:
document = list(range(0, 100))

In [ ]:
window_size = 10
start = 0
step = 5

chunks = []

while start < len(document):
    end = start + window_size
    chunk = document[start:end]
    if len(chunk) < window_size:
        break
    chunks.append(chunk)
    print(chunk)
    start = start + step

## The same window over text

The loop wrapped up as a function, with `start` recorded on each chunk so a result can point back
at where in the page it came from. `step` keeps the meaning it had in the list above - how far the
window moves - so the overlap is `size - step`. Two differences from the list version: the short
tail at the end is kept rather than dropped, and `size`/`step` are validated, because `step >= size`
would otherwise leave `start` standing still and loop forever.

In [ ]:
def sliding_window(text, size=1000, step=500):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    chunks = []
    text_length = len(text)

    for start in range(0, text_length, step):
        chunk = text[start:start + size]
        chunks.append({"start": start, "content": chunk})

        if start + size > text_length:
            break

    return chunks

In [ ]:
sample_page = documents[10]["content"]
len(sliding_window(sample_page, size=3000, step=1500))

## Chunk every document

Pop `content` off a copy of each document, chunk it, then merge the metadata back into every chunk,
so a retrieved chunk still knows its title and filename. The chunk's own `start` and `content` are
merged last so a page whose frontmatter happens to define those keys can't overwrite them.
`chunk_id` records which slice of the page it came from.

In [ ]:
document_chunks = []

for doc in documents:
    if not doc.get("content"):
        continue
    copy = doc.copy()
    content = copy.pop("content")

    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        enriched = {**copy, **chunk}
        enriched["chunk_id"] = i
        document_chunks.append(enriched)

In [ ]:
document_chunks[10]

## Index the chunks instead

The same `Index` as before, fitted on chunks rather than whole pages. `fit` builds in memory, so
this is a second index - the page-level one is still there to compare against.

In [ ]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [ ]:
chunk_results = chunk_index.search(query)

## Same job, done by the library

`chunk_documents` is the `gitsource` version of the loop above: same sliding window, same `step`
convention, same metadata merged into each chunk. Same pattern as the reader - do it by hand once
to know what it does, then use this. One gap: it doesn't add `chunk_id`, and it leaves the index
built two cells up pointing at the hand-made chunks.

In [ ]:
from gitsource import chunk_documents

In [ ]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

In [ ]:
document_chunks[10]

In [ ]:
len(document_chunks)

In [ ]:
from itertools import groupby

for _, page_chunks in groupby(document_chunks, key=lambda c: c["filename"]):
    for i, chunk in enumerate(page_chunks):
        chunk["chunk_id"] = i

chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [ ]:
chunk_results = chunk_index.search(query)
len(chunk_results)

# RAG

Goal: answer the question instead of handing back a list of chunks. Retrieval stays exactly as it
is above - same `chunk_index`, same search - and a model turns what it returns into prose.
Retrieve, stuff the results into a prompt, generate.

## Load the API key

`.env` sits at the repo root, two levels up from this folder, and `load_dotenv` reads it into the
environment. The check prints whether the key arrived and how long it is - never the key itself,
because this repo is public.

In [ ]:
import os
from dotenv import load_dotenv

# .env lives at the repo root, two levels up from this notebook
load_dotenv("../../.env")

key = os.environ.get("OPENAI_API_KEY")
# never print the key itself - this repo is public
print("key loaded:", bool(key), "| length:", len(key) if key else 0)

## Create the client

`OpenAI()` picks `OPENAI_API_KEY` up from the environment on its own, so the constructor takes no
arguments. The cell above is what makes that work - without it this is where things would break.

In [ ]:
from openai import OpenAI

openai_client = OpenAI()

## Retrieve the context

The retrieval half of RAG, unchanged: `chunk_index.search` with `num_results=5`, because five
chunks of 3000 characters is about as much context as is worth pasting into a prompt. The results
come back as dicts, so `json.dumps` flattens them into one string the model can read, metadata
and all.

Watch the order of the four cells below: the search runs while `query` is still
`"LLM as a Judge"` and only then gets reassigned, so `search_result` belongs to the old query.
Harmless here - both ask the same thing - but re-run the search after changing `query`.

In [ ]:
search_result = chunk_index.search(query, num_results=5)

In [ ]:
query = 'how do I implement llm as a judge?'

In [ ]:
import json

In [ ]:
search_result_json = json.dumps(search_result, indent=2)

## Build the prompt

Two parts doing different jobs. `instructions` is the system message - who the model is and what
it's for - while `user_prompt` carries this turn's question and the retrieved chunks. The
XML-ish tags aren't magic; they just mark where the question ends and the context begins, so the
model doesn't read one as part of the other.

In [ ]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the 
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

## Call the model

`llm` is the only place that talks to OpenAI: it assembles `messages`, skipping the system entry
when `instructions` is `None`, and returns `output_text` so no caller has to unpack a response
object. `gpt-4o-mini` is the default on purpose - this is a smoke test, not a quality contest.

In [ ]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })
    
    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [ ]:
answer = llm(user_prompt, instructions)

In [ ]:
print(answer)

## The same three steps as functions

Nothing new runs below - it's the cells above given names: `search` retrieves, `build_prompt`
formats, `llm` generates, and `rag` chains them. Worth the trouble because `rag(query)` is now
the whole pipeline in one call, and any step can be swapped without touching the others.

One gap to know about: `build_prompt` takes `search_results` but serialises `search_result`, the
global left over from the manual run, so it returns the same context whatever it is handed.

In [ ]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [ ]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the 
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_result, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## Run it end to end

Retrieve, prompt and generate in one call - the same answer as above, without the intermediate
names.

In [ ]:
rag('how do I implement llm as a judge?')